# Microsoft Foundry — notebook 2 · agents you host, and the plumbing around them

Companion to notebook 1 and to slides 4, 14, 17, 18 and 21 of the deck. Notebook 1 called platform features from Python; this one **interacts with things you deployed beforehand**: two hosted agents, their telemetry, and an AI gateway in front of a model deployment.

**Do the day before** (details in `agent/README.md`)
1. `cd agent && azd up` — deploys `demo-hosted-agent` (Agent Framework, Responses protocol) and `demo-long-running-agent` (resilient + steerable, preview) to your existing Foundry project.
2. Application Insights connected to the project (already true if you ran notebook 1).
3. **AI gateway**: an Azure API Management instance associated with your Foundry resource, with your chat deployment imported as an API and a token-limit policy on it (step 4 explains the smallest setup).
4. `.env` next to this notebook with `FOUNDRY_PROJECT_ENDPOINT`, `FOUNDRY_MODEL_NAME`, `HOSTED_AGENT_NAME`, `LONG_RUNNING_AGENT_NAME`, `APIM_GATEWAY_URL`, `APIM_SUBSCRIPTION_KEY`.

Lines marked `# verify` use a documented shape that was not executed here — check them against the linked Learn page before presenting. Long-running agents and the AI gateway in Foundry are **preview**.

## 0 · Connect  *(slide 5)*
Same two clients as notebook 1. Hosted agents expose a dedicated Responses endpoint the moment they are deployed — publishing is not required — and any OpenAI-compatible SDK can call it.

Docs: learn.microsoft.com/azure/foundry/agents/concepts/hosted-agents

In [1]:
import os, time, json
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()
PROJECT_ENDPOINT  = os.getenv("FOUNDRY_PROJECT_ENDPOINT", "https://<your-foundry-account>.services.ai.azure.com/api/projects/<your-project>")
CHAT_DEPLOYMENT   = os.getenv("FOUNDRY_MODEL_NAME", "<chat-model-deployment-name>")
HOSTED_AGENT      = os.getenv("HOSTED_AGENT_NAME", "demo-hosted-agent")
LONG_RUNNING      = os.getenv("LONG_RUNNING_AGENT_NAME", "demo-long-running-agent")

credential = DefaultAzureCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client = project.get_openai_client()

def agent_responses_endpoint(name):
    # documented pattern: {project_endpoint}/agents/{name}/endpoint/protocols/openai/responses
    return f"{PROJECT_ENDPOINT}/agents/{name}/endpoint/protocols/openai/responses"

def bearer():
    return credential.get_token("https://ai.azure.com/.default").token

print("Hosted agent endpoint      :", agent_responses_endpoint(HOSTED_AGENT))
print("Long-running agent endpoint:", agent_responses_endpoint(LONG_RUNNING))

Hosted agent endpoint      : https://cog-tb7tpjtuee4ji.services.ai.azure.com/api/projects/cog-tb7tpjtuee4ji-project/agents/demo-hosted-agent/endpoint/protocols/openai/responses
Long-running agent endpoint: https://cog-tb7tpjtuee4ji.services.ai.azure.com/api/projects/cog-tb7tpjtuee4ji-project/agents/demo-long-running-agent/endpoint/protocols/openai/responses


## 1 · Hosted agent — your code, Foundry's runtime  *(slide 14)*

`demo-hosted-agent` is an Agent Framework agent with one function tool, running in a per-session VM-isolated sandbox with its **own Microsoft Entra agent identity**.

Hosted agents must be called through their **dedicated Responses endpoint** with a bearer token. The project's OpenAI client with an `agent_reference` is not supported for these hosted agents; that route returns HTTP 400.

The next two cells call the dedicated endpoint with different prompts. Then use the project client to look up the agent's identity and versions in the inventory.

In [2]:
import requests

resp = requests.post(
    agent_responses_endpoint(HOSTED_AGENT),
    params={"api-version": "v1"},
    headers={"Authorization": f"Bearer {bearer()}", "Content-Type": "application/json"},
    json={"input": "Which deployment type should I use for an interactive chat assistant in Switzerland?"},
    timeout=180,
 )
resp.raise_for_status()
body = resp.json()
output = body.get("output", [])
print("\n".join(
    content["text"]
    for item in output if item.get("type") == "message"
    for content in item.get("content", []) if content.get("type") == "output_text"
))
print("\nOutput items:", [item.get("type") for item in output])

The deployment hint recommends: “Provisioned (PTU) for guaranteed throughput; or GlobalStandard with Priority Processing on pay-as-you-go.”
For your interactive chat assistant, consider pay-as-you-go for variable demand or PTU for predictable, sustained traffic, subject to model and regional availability.
If prompts and responses must remain in Switzerland, avoid Global Standard and choose a regional deployment in a supported Swiss region—confirm residency requirements before deciding.

Output items: ['function_call', 'function_call_output', 'reasoning', 'message']


In [3]:
import requests

# Same agent, called on its dedicated endpoint — what an external app or another agent would do.
url = agent_responses_endpoint(HOSTED_AGENT)
resp = requests.post(url, params={"api-version": "v1"},
                     headers={"Authorization": f"Bearer {bearer()}", "Content-Type": "application/json"},
                     json={"input": "In one sentence, what are you and where are you running?"})
resp.raise_for_status()
body = resp.json()
print(body.get("status"), "—", next((o["content"][0]["text"] for o in body.get("output", []) if o.get("type") == "message"), body))

completed — I’m an AI assistant serving as a Microsoft Foundry architect, accessed through an API; I don’t have visibility into the infrastructure or region hosting this session.


In [4]:
# Inventory: versions, identity, protocols
versions = list(project.agents.list_versions(agent_name=HOSTED_AGENT))      # verify: list_versions signature in your SDK version
latest = versions[0]
print("Agent   :", latest.name, "version", latest.version)
print("Kind    :", getattr(latest.definition, "kind", "hosted"))
print("Identity:", getattr(latest, "agent_identity", None) or "see Control Plane → Assets → Entra ID column")

Agent   : demo-hosted-agent version 2
Kind    : hosted
Identity: see Control Plane → Assets → Entra ID column


## 2 · Procurement decision brief — background, resilient, steerable *(slide 17)*

**Business scenario:** a Swiss manufacturer must choose a sourcing strategy before a product launch in six months. The procurement manager needs a risk assessment, a mitigation plan with owners, and an executive decision brief. Each section is produced by a real model call using the preceding sections.

**Why a long-running agent?** Multi-stage analysis and review can outlast a chat request. Background execution lets the manager leave and reconnect; checkpointed sections survive a process restart; steering stops work on an obsolete decision when priorities change. A single short summary would be better served by a normal model call.

1. **Start:** prepare a preliminary battery-module sourcing brief. Create a conversation on the hosted agent's own endpoint and submit a stored background response.
2. **Steer (optional):** promptly redirect the same conversation to battery-recycling partner selection. The original turn winds down and a separate brief begins; previous-topic conclusions are not reused.
3. **Read the deliverable:** the final cell renders each section as it becomes available. Skip steering to finish the original brief. Polling can reconnect to the same response.
4. **Recover (local test):** set `SIMULATE_CRASH_AFTER_STAGE=0`, run locally, and restart after the first checkpoint. Completed sections retain their content and IDs; only uncommitted work is repeated.

**Boundaries:** this is a preliminary, model-generated brief, not verified supplier due diligence. No live supplier, price or regulatory sources are connected. Assumptions and missing evidence must be identified; a human approves decisions. A full brief normally makes three billable model calls, with additional usage possible for retries or interrupted work.

The hosted identity needs model-invocation access and `AZURE_AI_MODEL_DEPLOYMENT_NAME`. Redeploy after the refactor and start a fresh conversation. The manifest adds a **10-second demo-only delay per stage** to allow steering; set `DEMO_STAGE_DELAY_SECONDS` to `0` for normal use. Actual model latency varies.

Docs: https://learn.microsoft.com/azure/foundry/agents/how-to/deploy-resilient-agent · https://learn.microsoft.com/azure/foundry/agents/how-to/deploy-steerable-agent

In [17]:
from IPython.display import Markdown, display

LR_URL = agent_responses_endpoint(LONG_RUNNING)
API_VERSION = "2025-11-15-preview"
H = lambda: {"Authorization": f"Bearer {bearer()}", "Content-Type": "application/json"}
steer = None

def agent_request(method, url, *, required_fields=("id", "status"), **kwargs):
    response = requests.request(
        method, url, params={"api-version": API_VERSION}, headers=H(),
        timeout=180, **kwargs,
    )
    if not response.ok:
        raise requests.HTTPError(
            f"{method} {url}: HTTP {response.status_code}\n{response.text}",
            response=response,
        )
    payload = response.json()
    if not isinstance(payload, dict) or any(field not in payload for field in required_fields):
        raise ValueError(f"Unexpected agent response: {payload!r}")
    return payload

def briefing_markdown(response):
    sections = [
        content["text"]
        for item in response.get("output", []) if item.get("type") == "message"
        for content in item.get("content", []) if content.get("type") == "output_text"
    ]
    completed = sum(section.startswith(("## Risk assessment\n", "## Mitigation plan\n", "## Decision brief\n")) for section in sections)
    parts = [f"**Status: {response['status']} | Briefing sections: {completed}/3**"]
    parts.extend(sections)
    if not sections:
        parts.append("The first model-backed section is being prepared; no results are available yet.")
    if response.get("error"):
        parts.append(f"**Error:** {response['error']}")
    return Markdown("\n\n---\n\n".join(parts))

def show(rid):
    response = agent_request("GET", f"{LR_URL}/{rid}")
    display(briefing_markdown(response))
    return response

conversation = agent_request(
    "POST", f"{LR_URL.removesuffix('/responses')}/conversations",
    required_fields=("id",), json={},
)
conv_id = conversation["id"]
start = agent_request(
    "POST", LR_URL,
    json={
        "input": (
            "Prepare a preliminary sourcing decision brief for a Swiss manufacturer launching "
            "a stationary battery-storage product in six months. Compare single-source and "
            "dual-source battery-module sourcing. Priorities: continuity of supply, EU/Swiss "
            "logistics and traceability. No supplier quotes, contracts or audited data have "
            "been provided. Include risks, mitigation owners and evidence needed before approval."
        ),
        "conversation": conv_id, "store": True, "background": True,
    },
)
resp_id = start["id"]
print("Started briefing:", resp_id, "| Conversation:", conv_id)
time.sleep(3)
response = show(resp_id)

Started briefing: caresp_068d5625390c6dcc00mYRui7cTbQKmBUODLR1kyZDqffMt6e3i | Conversation: conv_068d5625390c6dcc00IqjU867nIGRPt4cfsMxuJCaTYTXlGmaV


**Status: in_progress | Briefing sections: 0/3**

---

The first model-backed section is being prepared; no results are available yet.

In [18]:
if not isinstance(conv_id, str) or not conv_id:
    raise ValueError("Run the preceding background-response cell first to create a conversation.")

first_turn = agent_request("GET", f"{LR_URL}/{resp_id}")
if first_turn["status"] not in ("queued", "in_progress"):
    raise RuntimeError("The first turn has already stopped. Rerun the preceding cell, then steer promptly.")

steer = agent_request(
    "POST", LR_URL,
    json={
        "input": (
            "Change of priority: prepare a preliminary decision brief for the same Swiss "
            "manufacturer on selecting a battery-recycling partner, not battery-module sourcing. "
            "The launch is still in six months. Compare one regional partner versus a "
            "primary partner with a backup. Focus on traceability, continuity and evidence "
            "needed to assess EU/Swiss waste-shipment obligations. No supplier quotes, permits "
            "or audited data are available. Give mitigation owners and keep the final executive "
            "brief under 200 words. Do not assume regulatory compliance."
        ),
        "conversation": conv_id, "store": True, "background": True,
    },
)
print("Redirected briefing accepted:", steer["id"], "| Status:", steer["status"])
time.sleep(3)
print("Original sourcing brief:")
first_turn = show(resp_id)
print("Recycling-partner brief:")
response = show(steer["id"])

Redirected briefing accepted: caresp_068d5625390c6dcc00rwT65bFWIzwcXEOli89J3o8tM6mu9LCo | Status: queued
Original sourcing brief:


**Status: completed | Briefing sections: 2/3**

---

## Risk assessment

# Preliminary sourcing decision brief

**Status:** For human approval; not verified due diligence.

## Decision and constraints
Select single-source or dual-source battery-module sourcing for a Swiss manufacturer launching stationary battery storage in **six months**.

**Supplied facts:** Priorities are supply continuity, EU/Swiss logistics and traceability. No supplier quotes, contracts or audited data are available.

**Preliminary direction:** Pursue dual-source qualification if both modules can meet product requirements within the launch window. Otherwise, consider a primary source with a qualified backup roadmap and evidence-based inventory contingency. Neither approach is approval-ready.

| Approach | Potential advantages | Constraints and trade-offs |
|---|---|---|
| Single-source | Simpler integration, qualification and traceability administration | Concentrated disruption exposure; continuity depends on demonstrated capacity and recovery arrangements |
| Dual-source | Alternative supply route and reduced supplier dependence | Additional qualification, integration and traceability work; shared upstream dependencies may negate resilience |

## Five prioritized risks

*Priority reflects stated objectives, not assessed likelihood.*

| Risk | Business impact | Evidence or assumption | Validation needed | Mitigation owner |
|---|---|---|---|---|
| **1. Supply interruption or inadequate capacity** | Launch delay or production stoppage | Supplier capacity and commitments unknown | Capacity evidence, allocation terms, lead times, recovery plans; map shared upstream dependencies | Procurement + Operations: qualify alternatives and contingency stock |
| **2. Qualification exceeds six months** | Launch slips; nominal backup cannot be used | Assumption: module differences may require redesign/testing | Module specifications, compatibility tests, qualification schedule and engineering resources | Engineering + Quality: establish qualification gates |
| **3. EU/Swiss logistics disruption** | Late deliveries, customs holds, inventory shortages | Routes and shipping arrangements unknown | Route plans, dangerous-goods documentation, customs responsibilities, transit evidence | Logistics: validate routes and fallback carriers |
| **4. Incomplete traceability** | Slow containment, broader recalls, customer exposure | No audited traceability data supplied | Demonstrate cell-to-module-to-finished-product genealogy and retrieval tests | Quality: require traceability controls |
| **5. Unverified safety/conformity readiness** | Approval delays, unsafe product, market-access constraints | Applicable requirements and supporting evidence unconfirmed | Market-specific requirements assessment, test reports and technical documentation review | Compliance + Engineering: close gaps before approval |

---

## Mitigation plan

# Preliminary sourcing decision brief

**Status:** For human approval; not verified due diligence.

**Supplied facts:** Swiss stationary-storage launch in six months; priorities are continuity, EU/Swiss logistics and traceability. No supplier quotes, contracts or audited data provided.

## Options and provisional direction

| Option | Practical benefits | Main risks and trade-offs |
|---|---|---|
| **Single source** | Simpler integration, qualification and genealogy management | Supplier outage or allocation shortfall stops supply; contingency inventory ties up capital and cannot cover prolonged disruption. |
| **Dual source** | Potential alternative capacity and delivery routes | Additional testing, engineering and traceability complexity; shared cell suppliers or transport bottlenecks can undermine resilience. Interchangeability is unproven. |

**Recommendation:** Pursue parallel qualification now, but gate dual sourcing on demonstrated compatibility, independent supply resilience and launch readiness. If only one source qualifies, consider a controlled single-source launch with contingency stock and a dated backup-qualification plan. An unqualified backup provides no immediate continuity.

## Mitigations and decision gates

*Timing is indicative, not a confirmed qualification schedule.*

| Timing / gate | Accountable role | Required action and gate condition |
|---|---|---|
| **Weeks 1–2: feasibility** | Engineering lead | Define module interfaces, safety/performance requirements and qualification resources; decide whether two sources can realistically qualify. |
| **Weeks 2–6: supply resilience** | Procurement lead | Validate capacity, allocation, lead times, recovery arrangements and common upstream dependencies; propose inventory coverage from substantiated scenarios. |
| **Months 2–4: operational validation** | Logistics lead | Validate EU/Swiss routes, fallback carriers, dangerous-goods handling and customs responsibilities through documentation and trial shipments. |
| **Months 2–5: qualification** | Quality lead | Complete engineering tests and genealogy retrieval exercise; Compliance lead reviews market-specific requirements and supporting documentation. |
| **Before launch: approval** | Human sourcing authority | Review closed gaps, residual risks, commercial terms and contingency readiness; defer approval if critical evidence remains missing. |

**Evidence still needed:** Supplier quotations and proposed contracts; capacity records; specifications and samples; test reports; requirements assessment; logistics documentation; traceability records and audit access.

---

Work stopped before the next section was committed. Completed sections above are preliminary, not a final decision brief. A queued steering request will run separately.

Recycling-partner brief:


**Status: in_progress | Briefing sections: 0/3**

---

The first model-backed section is being prepared; no results are available yet.

In [19]:
target_id = steer["id"] if steer else resp_id
deadline = time.monotonic() + 600
progress = display(Markdown("Waiting for briefing sections..."), display_id=True)
while True:
    response = agent_request("GET", f"{LR_URL}/{target_id}")
    progress.update(briefing_markdown(response))
    if response["status"] in ("completed", "failed", "cancelled", "incomplete"):
        break
    if time.monotonic() >= deadline:
        print("The agent is still running. Rerun this cell to reconnect to the same briefing.")
        break
    time.sleep(5)
if response["status"] in ("failed", "cancelled", "incomplete"):
    raise RuntimeError(f"Briefing {response['status']}: {response.get('error') or response.get('incomplete_details') or 'No additional details'}")

**Status: completed | Briefing sections: 3/3**

---

## Risk assessment

## Preliminary decision brief
*Not verified due diligence; supplier selection requires human approval.*

**Decision:** Select a battery-recycling arrangement for a Swiss manufacturer launching in six months.

**Known constraints:** No supplier quotes, permits or audited data are available. Battery classification, shipment routes and partner capabilities remain evidence gaps; regulatory compliance is not assumed.

**Options:** One regional partner simplifies coordination but concentrates disruption risk; proximity alone does not establish compliance. A primary partner with a backup offers potential continuity but requires separately validated capacity, traceability and shipment arrangements.

| Prioritized risk | Business impact | Evidence or assumption | Validation needed / mitigation owner |
|---|---|---|---|
| 1. Waste-shipment obligations | Blocked shipments; launch delay | Routes, classification and permits unknown | **Legal/EHS:** establish waste classification, origin/transit/destination rules, required consents and documentation |
| 2. Traceability gaps | Unsubstantiated recycling outcomes | No audited data | **Quality/EHS:** test chain-of-custody records, downstream destinations and mass-balance evidence |
| 3. Partner interruption | Waste accumulation; production disruption | Single partner concentrates exposure | **Operations:** verify capacity, outage plans and safe storage limits |
| 4. Unusable backup | Continuity benefit fails | Backup readiness unproven | **Procurement/EHS:** validate independent capacity, activation terms and shipment arrangements |
| 5. Launch readiness | Schedule or budget overrun | Six months; no quotes | **Procurement:** obtain comparable proposals and qualification milestones |

**Provisional direction:** Evaluate primary-plus-backup, conditional on evidence and launch feasibility.

---

## Mitigation plan

## Preliminary decision brief
*Not verified due diligence; selection requires human approval.*

**Supplied facts:** Swiss manufacturer; launch in six months; no supplier quotes, permits or audited data. Classification, routes and capabilities remain unverified; compliance is not assumed.

| Option | Practical trade-off |
|---|---|
| **One regional partner** | Simpler traceability oversight and coordination, but concentrated outage risk. Proximity does not establish lawful shipments. |
| **Primary plus backup** | Potentially stronger continuity, but additional qualification effort. Benefit depends on independently available capacity and usable shipment routes. |

**Provisional direction:** Evaluate primary-plus-backup; retain single-partner sourcing only if continuity safeguards are credible.

**Mitigations and decision gates**
- **Month 1 — Legal/EHS:** Establish battery/waste classification and origin, transit and destination routes; determine applicable EU/Swiss shipment obligations. **Gate:** documented requirements before route commitment.
- **Months 2–3 — Procurement/Quality/EHS:** Obtain quotes, relevant permits and scope, required consents, downstream destinations, chain-of-custody samples, mass-balance records and audit evidence. **Gate:** resolve material authorization and traceability gaps before shortlist approval.
- **Months 3–4 — Operations/Procurement:** Verify capacity, outage plans, safe storage limits and backup activation terms; identify shared failure points. **Gate:** credible continuity plan.
- **Months 5–6 — Quality/Legal/EHS:** Validate traceability through a lawful trial and confirm route-specific documentation. **Gate:** management authorizes launch only after critical gaps close; otherwise delay affected shipments.

---

## Decision brief

## Preliminary executive brief
*Not verified due diligence; partner selection requires human approval.*

**Supplied facts:** Swiss manufacturer; launch in six months; no supplier quotes, permits or audited data available. Regulatory compliance is not assumed.

**Options**
- **One regional partner:** Fewer interfaces may simplify traceability oversight, but concentrate interruption risk. Proximity establishes neither traceability nor lawful shipments.
- **Primary plus backup:** Potentially stronger continuity, with additional qualification and coordination effort. Benefits remain unproven unless backup capacity, downstream dependencies and shipment arrangements are independently assessed.

**Conditional recommendation:** Prefer primary-plus-backup if both partners and routes pass evidence-based qualification before launch. Otherwise consider one regional partner only with credible outage and storage safeguards; defer affected shipments if critical gaps remain.

**Three next actions**
1. **Legal/EHS — month 1:** Establish battery/waste classification and origin, transit and destination routes; identify applicable EU/Swiss obligations and evidence needed, including authorizations, consents and shipment documentation.
2. **Quality/EHS — months 2–3:** Obtain permit scopes, downstream destinations, chain-of-custody samples, mass-balance records and audit evidence; validate traceability.
3. **Procurement/Operations — months 3–5:** Obtain comparable quotes; verify capacity, shared failure points, safe storage limits and backup activation terms.

**Unresolved gaps:** Classification, routes, authorizations, recycling outcomes, capacity, costs and six-month qualification feasibility. None is verified.

## 3 · Telemetry — nothing to instrument here  *(slide 21)*

Both hosted agents already emit OpenTelemetry spans: the platform injects the Application Insights connection string into the container and the protocol libraries trace by default. Your notebook code adds nothing. This cell queries Application Insights for the spans the last two steps produced, so you can show the same data that backs **Foundry portal → Traces** and **Control Plane → Assets → Traces**.

Needs `pip install azure-monitor-query` and **Log Analytics Reader** on the Application Insights resource.

Docs: learn.microsoft.com/azure/foundry/observability/how-to/trace-agent-client-side

In [23]:
from datetime import timedelta
from azure.monitor.query import LogsQueryClient
from dotenv import dotenv_values, find_dotenv

import os
env_values = dotenv_values(find_dotenv(usecwd=True))
APP_INSIGHTS_RESOURCE_ID = (env_values.get("APP_INSIGHTS_RESOURCE_ID") or os.getenv("APP_INSIGHTS_RESOURCE_ID", "")).strip()
resource_parts = APP_INSIGHTS_RESOURCE_ID.strip("/").split("/")
if (
    len(resource_parts) != 8
    or resource_parts[0].lower() != "subscriptions"
    or resource_parts[2].lower() != "resourcegroups"
    or [part.lower() for part in resource_parts[4:7]] != ["providers", "microsoft.insights", "components"]
    or any(not part or "<" in part or ">" in part for part in resource_parts)
):
    raise ValueError(
        "APP_INSIGHTS_RESOURCE_ID must be the Application Insights component resource ID: "
        "/subscriptions/.../resourceGroups/.../providers/Microsoft.Insights/components/... "
        "Do not use the Foundry project's /connections/... ID."
    )

logs = LogsQueryClient(credential)
kql = f"""
union dependencies, requests
| where timestamp > ago(30m)
| where cloud_RoleName has "{HOSTED_AGENT}" or cloud_RoleName has "{LONG_RUNNING}" or name has "responses"
| project timestamp, cloud_RoleName, name, duration, success, operation_Id
| order by timestamp desc
| take 20
"""
result = logs.query_resource(APP_INSIGHTS_RESOURCE_ID, kql, timespan=timedelta(minutes=30))
row_count = 0
for table in result.tables:
    print(" | ".join(table.columns))
    for row in table.rows:
        print(row)
        row_count += 1
print(f"\nReturned {row_count} span(s).")
if row_count == 0:
    print("No matching spans in the last 30 minutes. Check the telemetry destination, filters and ingestion delay.")

timestamp | cloud_RoleName | name | duration | success | operation_Id
[datetime.datetime(2026, 9, 14, 13, 1, 25, 403334, tzinfo=<isodate.tzinfo.Utc object at 0x7fb5b5154a70>), 'demo-long-running-agent', 'GET /api/projects/cog-tb7tpjtuee4ji-project/tasks', 394, 'True', 'eb0564ccaeee92afcac94a608b15dd0a']
[datetime.datetime(2026, 9, 14, 13, 1, 25, 403334, tzinfo=<isodate.tzinfo.Utc object at 0x7fb5b5154a70>), 'demo-long-running-agent', 'GET /api/projects/cog-tb7tpjtuee4ji-project/tasks', 394, 'True', 'eb0564ccaeee92afcac94a608b15dd0a']
[datetime.datetime(2026, 9, 14, 12, 58, 39, 811943, tzinfo=<isodate.tzinfo.Utc object at 0x7fb5b5154a70>), 'demo-long-running-agent', 'GET /api/projects/cog-tb7tpjtuee4ji-project/storage/responses/caresp_068d5625390c6dcc00rwT65bFWIzwcXEOli89J3o8tM6mu9LCo', 75, 'True', '0ca2e64325fa8e2451a76610f1d06552']
[datetime.datetime(2026, 9, 14, 12, 58, 39, 811943, tzinfo=<isodate.tzinfo.Utc object at 0x7fb5b5154a70>), 'demo-long-running-agent', 'GET /api/projects/co

**Optional — a local OTLP view.** Run `docker run -d -p 18888:18888 -p 4317:18889 mcr.microsoft.com/dotnet/aspire-dashboard` and start an agent locally with `azd ai agent run --no-client` and `OTEL_EXPORTER_OTLP_ENDPOINT=http://localhost:4317`; the Aspire Dashboard at http://localhost:18888 shows the same spans live. `# verify` the exact OTLP environment variable the agent-server library honours in your version.

## 4 · AI gateway — token limits and cost control in two calls  *(slide 4 and 18)*

The smallest useful setup, done beforehand: an Azure API Management instance (Basic v2 is enough) **associated with the Foundry resource** (Foundry portal → Manage → AI Gateway, preview), your chat deployment imported as an API with managed-identity auth, and an `llm-token-limit` policy of, say, 500 tokens per minute per subscription key. Then:

1. call the model **through the gateway URL** until the limit trips — `429` with `Retry-After`;
2. show the token metric per consumer in Application Insights.

Docs: learn.microsoft.com/azure/api-management/genai-gateway-capabilities

In [ ]:
APIM_GATEWAY_URL     = os.getenv("APIM_GATEWAY_URL", "https://<your-apim>.azure-api.net/<api-suffix>")   # e.g. .../openai
APIM_SUBSCRIPTION_KEY = os.getenv("APIM_SUBSCRIPTION_KEY", "<subscription-key>")

from openai import OpenAI, RateLimitError

# Same OpenAI SDK, different base URL: API Management fronts the Foundry deployment.
gw = OpenAI(base_url=f"{APIM_GATEWAY_URL}/v1", api_key="not-used",                      # verify: path suffix of your imported API
            default_headers={"Ocp-Apim-Subscription-Key": APIM_SUBSCRIPTION_KEY})

total = 0
for i in range(1, 30):
    try:
        r = gw.responses.create(model=CHAT_DEPLOYMENT, input="List five Azure regions in Europe, one per line.")
        total += r.usage.total_tokens
        print(f"call {i:2d}  ok    tokens so far: {total}")
    except RateLimitError as e:
        retry = getattr(e, "response", None) and e.response.headers.get("Retry-After")
        print(f"call {i:2d}  429 — token limit reached after ~{total} tokens; Retry-After: {retry}s")
        break

In [ ]:
# Token usage per consumer, as emitted by the gateway (llm-emit-token-metric / built-in logging)
kql = """
customMetrics
| where timestamp > ago(30m)
| where name has "Token"                       // e.g. Prompt Tokens, Completion Tokens, Total Tokens
| summarize tokens = sum(valueSum) by name, tostring(customDimensions["Subscription ID"])
| order by tokens desc
"""
result = logs.query_resource(APP_INSIGHTS_RESOURCE_ID, kql, timespan=timedelta(minutes=30))   # verify: your gateway's App Insights resource
for table in result.tables:
    for row in table.rows: print(row)
print("\nControl Plane → Quota shows the same limits once the gateway is attached to the Foundry resource.")

## Close in the portal  *(slide 18)*

**Control Plane → Operate → Assets**: both hosted agents with their Entra Agent IDs, runs, token usage and estimated cost; **Traces** tab for the steering run; **Quota** for the gateway limits. One screen, four demos.

### Cleanup
`cd agent && azd down` removes what the `demo` environment created (the two agents), and leaves your Foundry project, deployments and API Management untouched.